In [1]:
!git clone https://github.com/tuananhpham-vnu/ADAPT.git

Cloning into 'ADAPT'...
remote: Enumerating objects: 1074, done.
remote: Counting objects: 100% (280/280), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 1074 (delta 131), reused 199 (delta 67), pack-reused 794 (from 1)
Receiving objects: 100% (1074/1074), 132.06 MiB | 48.54 MiB/s, done.
Resolving deltas: 100% (455/455), done.


In [2]:
%cd ADAPT/
!ls

/kaggle/working/ADAPT
adapt.ipynb	  embedder	   ReAct			 scripts
adapt_tracing.py  environment.yml  README.md			 src
agentdriver	  git		   requirements-agentdriver.txt  survey
algo		  _guidance	   requirements-aqua.txt	 tests
configs		  LICENSE	   requirements.txt
EhrAgent	  make.ps1	   sanity_check.py


In [3]:
%%capture
!pip install -r requirements.txt
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install "openai<1.0

In [4]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [5]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [6]:
!python algo/trigger_optimization.py --agent qa --algo ap --model dpr-ctx_encoder-single-nq-base --save_dir ./results --ppl_filter --target_gradient_guidance --asr_threshold 0.5 --num_adv_passage_tokens 10 --golden_trigger -w -p

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 23020010 (models-uet-edu-vn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/ADAPT/wandb/run-20260922_103317-csuuhfo9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run expert-microwave-11
wandb: ⭐️ View project at https://wandb.ai/models-uet-edu-vn/agentpoison
wandb: 🚀 View run at https://wandb.ai/models-uet-edu-vn/agentpoison/runs/csuuhfo9
wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
10:33:19 | httpx | INFO | HTTP Reque

* legacy_raw — trigger cũ + renderer cũ (nối thẳng trigger + " " + query)
* legacy_heading — vẫn trigger cũ, chỉ viết hoa + thêm :
* generic_language — chọn lại trong 8 seed chung
* grounded_language — thêm ứng viên About/Regarding <topic> từ nội dung train
* topic_heading — chỉ cụm chủ đề trích nguyên văn
* query_adaptive — sinh/chọn theo từng query đang đến (chỉ có ở per_query)


In [7]:
# !python -m src.agentpoison.phases optimize \
#   --agent qa \
#   --retriever-model dpr-ctx_encoder-single-nq-base \
#   --num-iter 5 \
#   --num-cand 20 \
#   --num-grad-iter 3 \
#   --opt-batch-size 16 \
#   --trigger-output results/triggers/qa-dpr-smoke.json


In [8]:
# !python -m json.tool results/triggers/qa-dpr-smoke.json

In [9]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [10]:
# !python -m src.agentpoison.phases optimize \
#   --agent qa \
#   --retriever-model dpr-ctx_encoder-single-nq-base \
#   --num-iter 1000 \
#   --num-cand 100 \
#   --num-grad-iter 30 \
#   --opt-batch-size 32 \
#   --trigger-output results/triggers/qa-dpr-ap.json

In [11]:
# RUN_DIR=kaggle/working/ADAPT/results/ssh_smoke_01
# TRIGGER=kaggle/working/ADAPT/results/triggers/qa-dpr-ap.json

In [12]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [13]:
# !python -m src.agentpoison.phases prepare \
#   --run-dir "$RUN_DIR" \
#   --trigger-file "$TRIGGER" \
#   --provider deepseek \
#   --device cuda \
#   --num-queries 1 \
#   --batch-size 16 \
#   --index ReAct/database/embeddings/agentpoison_dpr \
#   --repeats 1


In [14]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [15]:
# !python -m src.agentpoison.phases retrieve --run-dir "$RUN_DIR"

In [16]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [17]:
# !python -m src.agentpoison.phases infer \
#   --run-dir "$RUN_DIR" \
#   --provider deepseek

In [18]:
    # !python -m src.agentpoison.phases infer \
    #   --run-dir "$RUN_DIR" \
    #   --provider deepseek \
    #   --retry-errors

In [19]:
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.ipc_collect()

In [20]:
# !python -m src.agentpoison.phases evaluate --run-dir "$RUN_DIR"